In [6]:

import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations

# =========================================================
# 0) 실행 환경/입출력
# =========================================================
INPUT_PATH = Path("20191001_OUTPUT.csv")
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 1) 분석 기본 설정
# =========================================================
USE_COLS = [
    "start_dt",
    "arv_dt",
    "start_emd",
    "arv_emd",
    "start_arv_place_type",
    "mvmn_time_sum",
    "popl_cnt"
]

PRIORITY_GROUPS = [
    ["HW", "WH"],
    ["HE", "EH"],
    ["WE", "EW"],
    ["HH", "WW", "EE"]
]

TARGET_RATIO = 0.60
YEAR_FROM, YEAR_TO = 2022, 2024   # 실데이터 연도에 맞게 변경
FILTER_H_INCLUDED = True

LITERATURE_WEIGHTS = {
    "유입인구": 0.45,
    "평균이동시간": 0.35,
    "총이동시간": 0.20
}

REG_TARGET_COL = None
FEATURE_COLS = ["총이동인구수", "평균이동시간", "총이동시간"]
_EQUAL_WEIGHTS = {"유입인구": 1, "평균이동시간": 1, "총이동시간": 1}


# =========================================================
# 2) 공통 유틸 함수
# =========================================================
def load_data(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"지원하지 않는 파일 형식입니다: {suffix}")

def minmax(s: pd.Series) -> pd.Series:
    if s.max() == s.min():
        return pd.Series(0.0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def normalize_weights(w: dict) -> dict:
    total = sum(w.values())
    if total == 0:
        n = len(w)
        return {k: 1 / n for k in w}
    return {k: v / total for k, v in w.items()}

def rank_spearman(df_a, df_b, key="자치구코드", col="출퇴근스트레스점수"):
    a = df_a[[key, col]].rename(columns={col: "a"})
    b = df_b[[key, col]].rename(columns={col: "b"})
    m = a.merge(b, on=key, how="inner")
    if len(m) < 2:
        return np.nan
    return m["a"].rank().corr(m["b"].rank(), method="pearson")

def topk_overlap(df_a, df_b, k=5, key="자치구코드", col="출퇴근스트레스점수"):
    sa = set(df_a.sort_values(col, ascending=False).head(k)[key])
    sb = set(df_b.sort_values(col, ascending=False).head(k)[key])
    return len(sa & sb) / k if k > 0 else np.nan


# =========================================================
# 3) 전처리
# =========================================================
def preprocess_movement(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df[USE_COLS].copy()

    df["start_dt"] = pd.to_datetime(df["start_dt"], errors="coerce")
    df["arv_dt"]   = pd.to_datetime(df["arv_dt"],   errors="coerce")

    df["start_emd"] = df["start_emd"].astype(str).str.strip()
    df["arv_emd"]   = df["arv_emd"].astype(str).str.strip()
    df["start_arv_place_type"] = (
        df["start_arv_place_type"].astype(str).str.upper().str.strip()
    )

    df["mvmn_time_sum"] = pd.to_numeric(df["mvmn_time_sum"], errors="coerce")
    df["popl_cnt"]      = pd.to_numeric(df["popl_cnt"],      errors="coerce")

    df = df.dropna(subset=["start_dt", "mvmn_time_sum", "popl_cnt"]).copy()

    # 연도 필터 — 범위 밖이면 경고 후 전체 사용
    filtered = df[df["start_dt"].dt.year.between(YEAR_FROM, YEAR_TO)].copy()
    if len(filtered) == 0:
        actual_years = sorted(df["start_dt"].dt.year.dropna().unique().tolist())
        print(
            f"[WARN] YEAR_FROM={YEAR_FROM}, YEAR_TO={YEAR_TO} 범위에 해당하는 데이터가 없습니다. "
            f"데이터 실제 연도: {actual_years}. 연도 필터를 건너뛰고 전체 데이터를 사용합니다."
        )
        filtered = df.copy()

    filtered["연도"] = filtered["start_dt"].dt.year
    filtered["출발_자치구코드"] = filtered["start_emd"].str[:5]
    filtered["도착_자치구코드"] = filtered["arv_emd"].str[:5]

    if FILTER_H_INCLUDED:
        filtered = filtered[
            filtered["start_arv_place_type"].str.contains("H", na=False)
        ].copy()

    return filtered


# =========================================================
# 4) 우선순위 유형 선택 (60% 룰)
# =========================================================
def select_place_types_by_priority(df: pd.DataFrame, target_ratio=0.60):
    total_pop = df["popl_cnt"].sum()
    selected_types = []
    selected_pop = 0.0

    for i, group in enumerate(PRIORITY_GROUPS, start=1):
        group_pop = df.loc[df["start_arv_place_type"].isin(group), "popl_cnt"].sum()
        selected_types.extend(group)
        selected_pop += group_pop

        ratio = selected_pop / total_pop if total_pop > 0 else 0.0
        print(f"[{i}순위] {group} 추가 | 누적 커버리지 {ratio:.2%}")

        if ratio >= target_ratio:
            break

    filtered_df = df[df["start_arv_place_type"].isin(selected_types)].copy()
    coverage = selected_pop / total_pop if total_pop > 0 else 0.0
    return filtered_df, selected_types, coverage


# =========================================================
# 5) 자치구 피처 생성
# =========================================================
def make_features(df: pd.DataFrame, group_col="도착_자치구코드") -> pd.DataFrame:
    f = (
        df.groupby(group_col, dropna=False)
        .agg(
            총이동인구수=("popl_cnt", "sum"),
            평균이동시간=("mvmn_time_sum", "mean")
        )
        .reset_index()
        .rename(columns={group_col: "자치구코드"})
    )
    f["총이동시간"] = f["총이동인구수"] * f["평균이동시간"]
    return f

def make_features_by_year(df: pd.DataFrame, group_col="도착_자치구코드") -> pd.DataFrame:
    f = (
        df.groupby(["연도", group_col], dropna=False)
        .agg(
            총이동인구수=("popl_cnt", "sum"),
            평균이동시간=("mvmn_time_sum", "mean")
        )
        .reset_index()
        .rename(columns={group_col: "자치구코드"})
    )
    f["총이동시간"] = f["총이동인구수"] * f["평균이동시간"]
    return f


# =========================================================
# 6) 데이터 기반 가중치 생성
# =========================================================
def pca_weights(feature_df: pd.DataFrame) -> dict:
    """PC1 적재값 비율로 가중치 생성. 행 부족(<3) 또는 수렴 실패 시 균등 가중치."""
    if len(feature_df) < 3:
        print(f"[WARN] pca_weights: 행 수 {len(feature_df)}개 부족 → 균등 가중치 적용")
        return normalize_weights(_EQUAL_WEIGHTS)

    X = feature_df[FEATURE_COLS].astype(float)
    X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    X = X.fillna(0.0)

    try:
        cov = np.cov(X.values, rowvar=False)
        eigvals, eigvecs = np.linalg.eigh(cov)
        pc1 = eigvecs[:, np.argmax(eigvals)]
        load = np.abs(pc1)
    except np.linalg.LinAlgError:
        print("[WARN] pca_weights: 고유값 수렴 실패 → 균등 가중치 적용")
        return normalize_weights(_EQUAL_WEIGHTS)

    w = {"유입인구": load[0], "평균이동시간": load[1], "총이동시간": load[2]}
    return normalize_weights(w)

def critic_weights(feature_df: pd.DataFrame) -> dict:
    """CRITIC 방식 가중치. 행 부족(<3) 시 균등 가중치."""
    if len(feature_df) < 3:
        print(f"[WARN] critic_weights: 행 수 {len(feature_df)}개 부족 → 균등 가중치 적용")
        return normalize_weights(_EQUAL_WEIGHTS)

    Z = feature_df[FEATURE_COLS].astype(float)
    Z = (Z - Z.min()) / (Z.max() - Z.min())
    Z = Z.fillna(0.0)

    std = Z.std(ddof=0)
    corr = Z.corr().fillna(0.0)

    cvals = {}
    for c in FEATURE_COLS:
        cvals[c] = std[c] * (1 - corr[c]).sum()

    w = {
        "유입인구":     cvals["총이동인구수"],
        "평균이동시간": cvals["평균이동시간"],
        "총이동시간":   cvals["총이동시간"]
    }
    return normalize_weights(w)

def regression_weights(feature_df: pd.DataFrame, target_col: str | None) -> dict | None:
    if not target_col or target_col not in feature_df.columns:
        return None

    d = feature_df.dropna(subset=FEATURE_COLS + [target_col]).copy()
    if len(d) < 5:
        return None

    X = d[FEATURE_COLS].astype(float)
    y = pd.to_numeric(d[target_col], errors="coerce")
    valid = y.notna()
    X, y = X[valid], y[valid]
    if len(X) < 5:
        return None

    Xz = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    Xz = Xz.fillna(0.0)
    yz = (y - y.mean()) / (y.std(ddof=0) if y.std(ddof=0) != 0 else 1.0)

    beta, *_ = np.linalg.lstsq(Xz.values, yz.values, rcond=None)
    beta = np.abs(beta)

    w = {"유입인구": beta[0], "평균이동시간": beta[1], "총이동시간": beta[2]}
    return normalize_weights(w)


# =========================================================
# 7) 점수 계산
# =========================================================
def score_with_weights(feature_df: pd.DataFrame, weights: dict, model_name: str) -> pd.DataFrame:
    d = feature_df.copy()

    d["유입인구_norm"]     = minmax(d["총이동인구수"])
    d["평균이동시간_norm"] = minmax(d["평균이동시간"])
    d["총이동시간_norm"]   = minmax(d["총이동시간"])

    w = normalize_weights(weights)

    d["출퇴근스트레스점수"] = (
        d["유입인구_norm"]     * w["유입인구"] +
        d["평균이동시간_norm"] * w["평균이동시간"] +
        d["총이동시간_norm"]   * w["총이동시간"]
    ) * 100

    d["모형"] = model_name
    d["가중치_유입인구"]     = w["유입인구"]
    d["가중치_평균이동시간"] = w["평균이동시간"]
    d["가중치_총이동시간"]   = w["총이동시간"]

    return d.sort_values("출퇴근스트레스점수", ascending=False).reset_index(drop=True)


# =========================================================
# 8) 전체 실행 + 검증
# =========================================================
def run_pipeline_all(raw_df: pd.DataFrame):
    # 1) 전처리
    d = preprocess_movement(raw_df)
    print(f"[INFO] 전처리 후 행 수: {len(d):,}")

    # 2) 우선순위 필터
    d_sel, selected_types, coverage = select_place_types_by_priority(d, TARGET_RATIO)

    # 3) 피처 생성
    f = make_features(d_sel, "도착_자치구코드")

    # 4) 가중치 시나리오
    w_lit    = normalize_weights(LITERATURE_WEIGHTS)
    w_pca    = pca_weights(f)
    w_critic = critic_weights(f)
    w_reg    = regression_weights(f, REG_TARGET_COL)

    # 5) 모형별 점수
    results = [
        score_with_weights(f, w_lit,    "문헌기반"),
        score_with_weights(f, w_pca,    "PCA기반"),
        score_with_weights(f, w_critic, "CRITIC기반")
    ]
    if w_reg is not None:
        results.append(score_with_weights(f, w_reg, "회귀기반"))

    all_scores = pd.concat(results, ignore_index=True)

    # 6) 민감도 분석
    base = all_scores[all_scores["모형"] == "문헌기반"]
    sens_rows = []
    for m in all_scores["모형"].unique():
        cur = all_scores[all_scores["모형"] == m]
        sens_rows.append({
            "비교모형": m,
            "스피어만순위상관(문헌대비)": rank_spearman(base, cur),
            "Top5중복률(문헌대비)":       topk_overlap(base, cur, k=5)
        })
    sensitivity_df = pd.DataFrame(sens_rows)

    # 7) 연도 안정성 분석
    fy = make_features_by_year(d_sel, "도착_자치구코드")
    yearly_parts = []
    for y in sorted(fy["연도"].unique()):
        fy_y = fy[fy["연도"] == y].drop(columns=["연도"]).copy()
        sy = score_with_weights(fy_y, w_lit, f"문헌기반_{y}")
        sy["연도"] = y
        yearly_parts.append(sy[["연도", "자치구코드", "출퇴근스트레스점수"]])

    if yearly_parts:
        yearly_scores_df = pd.concat(yearly_parts, ignore_index=True)
    else:
        yearly_scores_df = pd.DataFrame(columns=["연도", "자치구코드", "출퇴근스트레스점수"])

    stability_rows = []
    years = sorted(yearly_scores_df["연도"].unique())
    for y1, y2 in combinations(years, 2):
        a = yearly_scores_df[yearly_scores_df["연도"] == y1]
        b = yearly_scores_df[yearly_scores_df["연도"] == y2]
        stability_rows.append({"연도쌍": f"{y1}-{y2}", "스피어만순위상관": rank_spearman(a, b)})
    stability_df = pd.DataFrame(stability_rows)

    # 8) 메타 정보
    meta_df = pd.DataFrame([{
        "분석기간":   f"{YEAR_FROM}~{YEAR_TO}",
        "H포함필터":  FILTER_H_INCLUDED,
        "선정유형":   ", ".join(selected_types),
        "커버리지":   coverage
    }])

    return {
        "전처리후_선정데이터":   d_sel,
        "모형별_점수결과":       all_scores,
        "민감도분석":            sensitivity_df,
        "연도안정성분석":        stability_df,
        "연도별점수_문헌기반":   yearly_scores_df,
        "메타정보":              meta_df
    }


# =========================================================
# 9) 실행/저장
# =========================================================
if __name__ == "__main__":
    print("[START] 데이터 로드")
    raw_df = load_data(INPUT_PATH)
    print(f"[INFO] 원본 행 수: {len(raw_df):,}")

    outputs = run_pipeline_all(raw_df)

    outputs["메타정보"].to_csv(            OUTPUT_DIR / "00_메타정보.csv",            index=False, encoding="utf-8-sig")
    outputs["전처리후_선정데이터"].to_csv( OUTPUT_DIR / "01_전처리후_선정데이터.csv", index=False, encoding="utf-8-sig")
    outputs["모형별_점수결과"].to_csv(     OUTPUT_DIR / "02_모형별_점수결과.csv",     index=False, encoding="utf-8-sig")
    outputs["민감도분석"].to_csv(          OUTPUT_DIR / "03_민감도분석.csv",          index=False, encoding="utf-8-sig")
    outputs["연도안정성분석"].to_csv(      OUTPUT_DIR / "04_연도안정성분석.csv",      index=False, encoding="utf-8-sig")
    outputs["연도별점수_문헌기반"].to_csv( OUTPUT_DIR / "05_연도별점수_문헌기반.csv", index=False, encoding="utf-8-sig")

    print("[DONE] 전체 파이프라인 완료")
    print(f"[INFO] 저장 경로: {OUTPUT_DIR.resolve()}")

[START] 데이터 로드
[INFO] 원본 행 수: 400
[WARN] YEAR_FROM=2022, YEAR_TO=2024 범위에 해당하는 데이터가 없습니다. 데이터 실제 연도: [2019]. 연도 필터를 건너뛰고 전체 데이터를 사용합니다.
[INFO] 전처리 후 행 수: 157
[1순위] ['HW', 'WH'] 추가 | 누적 커버리지 3.38%
[2순위] ['HE', 'EH'] 추가 | 누적 커버리지 94.69%
[DONE] 전체 파이프라인 완료
[INFO] 저장 경로: C:\8ssible-Healing-Seoul-Analysis\output
